# Smoke Test LLM Tahap 5

Jalankan hanya pada notebook Kaggle **privat** dengan Internet aktif dan GPU T4. Bundle berisi potongan laporan klinis dan tidak boleh dipublikasikan.

In [ ]:
from pathlib import Path
import zipfile

BUNDLE_ZIP = Path('/kaggle/input/cove-stage5/stage5_smoke_bundle.zip')
WORKDIR = Path('/kaggle/working/cove_stage5')
if not BUNDLE_ZIP.exists():
    raise FileNotFoundError(f'Sesuaikan BUNDLE_ZIP; tidak ditemukan: {BUNDLE_ZIP}')
WORKDIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(WORKDIR)
print('Bundle extracted to', WORKDIR)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(WORKDIR / 'notebooks/requirements-kaggle.txt')
], check=True)
print('Kaggle dependencies installed')

In [ ]:
import os

MODEL_KEYS = ['qwen25_7b_awq', 'mistral_7b_v03_bnb4']
RESULTS_DIR = WORKDIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
env = dict(os.environ)
env['PYTHONPATH'] = str(WORKDIR)
for model_key in MODEL_KEYS:
    jobs = WORKDIR / 'data/processed/llm_jobs' / f'stage5_smoke_{model_key}.jsonl'
    output = RESULTS_DIR / f'stage5_smoke_{model_key}_results.jsonl'
    command = [
        sys.executable, str(WORKDIR / 'notebooks/kaggle_runner.py'),
        '--config', str(WORKDIR / 'configs/llm.yaml'),
        '--jobs', str(jobs), '--output', str(output),
        '--model-key', model_key, '--backend', 'auto'
    ]
    print('Running', model_key)
    subprocess.run(command, check=True, env=env)
print('Both smoke-test runs completed')

In [ ]:
import json

for path in sorted(RESULTS_DIR.glob('*_results.jsonl')):
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]
    print('\n', path.name, 'rows=', len(rows))
    for row in rows:
        print(f"QA {row['qa_id']} | backend={row['backend']} | tokens={row['completion_tokens']}")
        print(row['text'][:500])